In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

In [2]:
L = 1.0  # Длина области (м)
Nx = 200  # Число узлов по пространству
Tm = 0.0  # Температура плавления (°C)
T_left = -15.0  # Температура слева (°C)
T_right = 5.0  # Температура справа (°C)
sim_time = 60.0  # Время симуляции (сек)

In [3]:
def thermal_conductivity(T):
    """Нелинейная теплопроводность: лёд/вода + линейная зависимость"""
    return np.where(T <= Tm, 2.2 + 0.01 * T, 0.56 + 0.001 * T)

In [4]:
def heat_capacity(T):
    """Теплоёмкость с пиком вблизи Tm"""
    delta_T = 0.5  # Ширина перехода
    return np.where(T < Tm - delta_T, 2100,
                    np.where(T > Tm + delta_T, 4200,
                             4200 + 2100 * (Tm - T) / delta_T))  # (Дж/(кг·K))

In [5]:
rho_ice = 917  # Плотность льда (кг/м³)
rho_water = 1000  # Плотность воды (кг/м³)
Lh = 334000  # Удельная теплота плавления (Дж/к

In [6]:
x = np.linspace(0, L, Nx)
dx = x[1] - x[0]

In [7]:
U = np.linspace(T_left, T_right, Nx)
s_pos = L / 2
s_idx = np.argmin(np.abs(x - s_pos))
U[:s_idx] = np.linspace(T_left, Tm, s_idx)
U[s_idx:] = np.linspace(Tm, T_right, Nx - s_idx)

In [8]:
U_history = [U.copy()]
front_positions = [s_pos]
front_velocities = [0.0]
times = [0.0]

In [9]:
dt = 0.1  # Начальный шаг по времени
current_time = 0

In [10]:
while current_time < sim_time:
    # 1. Определяем параметры на текущем шаге
    k = thermal_conductivity(U)
    cp = heat_capacity(U)
    rho = np.where(U <= Tm, rho_ice, rho_water)
    alpha = k / (rho * cp)  # Температуропроводность

    # 2. Строим матрицу для неявной схемы
    diagonals = np.zeros((3, Nx))
    diagonals[1, 1:-1] = 1 / dt + 2 * alpha[1:-1] / dx ** 2  # Главная диагональ
    diagonals[0, 2:] = -alpha[1:-1] / dx ** 2  # Верхняя диагональ
    diagonals[2, :-2] = -alpha[1:-1] / dx ** 2  # Нижняя диагональ

    # Граничные условия (Дирихле)
    diagonals[1, 0] = 1.0
    diagonals[1, -1] = 1.0
    A = diags(diagonals, [0, 1, -1], format='csc')

    # Правая часть
    b = U.copy()
    b[1:-1] = U[1:-1] / dt
    b[0] = T_left
    b[-1] = T_right

    # 3. Решаем систему
    U_new = spsolve(A, b)

    # 4. Находим положение фронта (интерполяция)
    crossing_idx = np.where(np.diff(np.sign(U_new - Tm)))[0]
    if len(crossing_idx) > 0:
        s_idx = crossing_idx[0]
        weight = (Tm - U_new[s_idx]) / (U_new[s_idx + 1] - U_new[s_idx])
        s_pos_new = x[s_idx] + weight * dx
    else:
        s_pos_new = s_pos

    # 5. Условие Стефана (расчёт скорости фронта)
    if 1 < s_idx < Nx - 2:
        # Градиенты температуры у фронта
        dT_left = (U_new[s_idx] - U_new[s_idx - 1]) / dx
        dT_right = (U_new[s_idx + 1] - U_new[s_idx]) / dx

        # Теплопроводности на фронте
        k_left = thermal_conductivity(Tm - 0.1)
        k_right = thermal_conductivity(Tm + 0.1)

        # Скорость фронта
        ds_dt = (k_left * dT_left - k_right * dT_right) / (rho_ice * Lh)
    else:
        ds_dt = 0.0

    # 6. Адаптивный шаг по времени
    if np.abs(ds_dt) > 1e-3:
        dt = min(0.1, 0.25 * dx / np.abs(ds_dt))

    # 7. Обновляем состояние
    U = U_new.copy()
    s_pos = s_pos_new
    current_time += dt

    # Сохраняем для анимации (каждые 0.5 сек)
    if current_time - times[-1] >= 0.5:
        U_history.append(U.copy())
        front_positions.append(s_pos)
        front_velocities.append(ds_dt)
        times.append(current_time)

D:\Users\Boris\AppData\Local\Temp\ipykernel_18472\1843594844.py:21: RuntimeWarning: overflow encountered in divide
  b[1:-1] = U[1:-1] / dt
D:\Users\Boris\AppData\Local\Temp\ipykernel_18472\1843594844.py:26: MatrixRankWarning: Matrix is exactly singular
  U_new = spsolve(A, b)

KeyboardInterrupt



In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), height_ratios=[3, 1])

# 1. График температуры
line, = ax1.plot([], [], 'b-', lw=2, label='Температура')
front_line = ax1.axvline(x=front_positions[0], color='r', linestyle='--', label='Фронт')
ax1.set_xlim(0, L)
ax1.set_ylim(T_left - 2, T_right + 2)
ax1.set_ylabel('Температура (°C)')
ax1.grid(True)
ax1.legend()

# 2. График положения и скорости фронта
front_pos_line, = ax2.plot([], [], 'g-', lw=2, label='Положение фронта')
ax2_twin = ax2.twinx()
front_vel_line, = ax2_twin.plot([], [], 'm--', lw=2, label='Скорость фронта')
ax2.set_xlim(0, sim_time)
ax2.set_ylim(0, L)
ax2.set_xlabel('Время (сек)')
ax2.set_ylabel('Положение (м)')
ax2_twin.set_ylabel('Скорость (м/сек)')
ax2.grid(True)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')


def init():
    line.set_data([], [])
    front_line.set_xdata([front_positions[0], front_positions[0]])
    front_pos_line.set_data([], [])
    front_vel_line.set_data([], [])
    return line, front_line, front_pos_line, front_vel_line


def animate(i):
    # Обновляем график температуры
    line.set_data(x, U_history[i])
    front_line.set_xdata([front_positions[i], front_positions[i]])

    # Обновляем график фронта
    front_pos_line.set_data(times[:i + 1], front_positions[:i + 1])
    front_vel_line.set_data(times[:i + 1], front_velocities[:i + 1])

    return line, front_line, front_pos_line, front_vel_line


ani = FuncAnimation(fig, animate, frames=len(U_history),
                    init_func=init, blit=False, interval=100)

plt.tight_layout()
plt.show()